In [ ]:
import pandas as pd
import numpy as np

import implicit
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares
from implicit.nearest_neighbours import bm25_weight
import hashlib

from scipy.sparse import coo_matrix
from implicit.evaluation import ranking_metrics_at_k

In [ ]:
df=pd.read_csv("Data/matriz.csv")

In [ ]:
# columna de codigos de categoria (para la matriz)
df["user_code"] = df["id_cliente_enc"].astype("category").cat.codes
df["item_code"] = df["cod_producto"].astype("category").cat.codes
df

In [ ]:
df.rename(columns={'id_cliente_enc': 'id'}, inplace=True)

In [ ]:
df['user_idx'] = df['user_code']
df['item_idx'] = df['item_code']

In [ ]:
matriz=df[["user_idx", "item_idx","cantidad"]]

Division train y test

In [ ]:
train=matriz.iloc[:int(0.8 * len(matriz))]
test=matriz.iloc[int(0.8 * len(matriz)):]

In [ ]:
#sparse matrix de interacciones usuario-item (orden correcto user item)    
def build_implicit_matrix(data):
    n_users = int(df['user_idx'].max() + 1)
    n_items = int(df['item_idx'].max() + 1)

    user_coords = data['user_idx'].values
    item_coords = data['item_idx'].values
    datos_matriz = np.ones(len(data))
    
    matrix = coo_matrix((datos_matriz, (user_coords, item_coords)), 
                        shape=(n_users, n_items))
    return matrix.tocsr()


train_matrix = build_implicit_matrix(train)
test_matrix = build_implicit_matrix(test)      

# asi las matrices tienen las dos el tamaño del dataset entero (sino da mal en ranking metrics)

In [ ]:
for k in [10, 20, 50, 100]:
    model_k = AlternatingLeastSquares(factors=k, regularization=0.01, iterations=40, alpha=20)
    model_k.fit(train_matrix)
    metricas_k = ranking_metrics_at_k(model_k, train_matrix, test_matrix, K=10)
    print(f"factors={k} → precision={metricas_k['precision']:.6f} | ndcg={metricas_k['ndcg']:.6f} | auc={metricas_k['auc']:.4f}")

In [ ]:
# modelo y entrenamos con la matriz de train
model=AlternatingLeastSquares(factors=50, regularization=0.01, iterations=40, alpha= 20, random_state=42)
model.fit(train_matrix)

Sacar los ids hasheados de los clientes a los que tenemos que recomendar

In [ ]:
obj=pd.read_csv("Data/objetivos_dni.csv")
obj

In [ ]:
obj=obj[obj["explica"]=="Otros como tu han comprado"].copy()
obj

In [ ]:
import hashlib

def anonimizar_dni(dni: str) -> str:
    return hashlib.sha256(dni.strip().upper().encode()).hexdigest()

obj["id"]=obj["id"].apply(anonimizar_dni)

obj.head(10)

In [ ]:
# los pongo en lista
lista_hashes=obj["id"].to_list()
lista_hashes

In [ ]:
merge=pd.merge(df,obj, on="id", how="inner")[["id", "user_code"]]
merge

In [ ]:
merge["user_code"].unique()

# OBJETIVO 2 -- recomendar a 10 clientes 

In [ ]:
lista_users=[4637,43465,43536,4984,24587,28386,9301,20490,7112,5349]

In [ ]:
codigo_real_items = df["cod_producto"].astype("category").cat.categories
idx2item = dict(enumerate(codigo_real_items))  #para conseguir el codigo real

codigo_real_users = df["id"].astype("category").cat.categories
user2idx = dict(zip(codigo_real_users, range(len(codigo_real_users))))

In [ ]:
full_matrix = build_implicit_matrix(df)

In [ ]:
print(len(lista_users))


In [ ]:
# Recomendaciones
resultados = []
for user_code in lista_users:
    user_idx = user_code
    ids, scores = model.recommend(
        user_idx, full_matrix[user_idx], N=1, filter_already_liked_items=True
    )
    resultados.append({
        'cliente': [user_code] * len(ids),
        'item_code': [idx2item[i] for i in ids],
        'score': scores
    })

df_recomendaciones = pd.DataFrame(resultados)
df_recomendaciones


In [ ]:
df_recomendaciones.dtypes

In [ ]:
for col in ["cliente", "item_code", "score"]:
    df_recomendaciones[col] = df_recomendaciones[col].astype(str).str.strip("[]")

In [ ]:
df_recomendaciones["cliente"] = pd.to_numeric(df_recomendaciones["cliente"])
df_recomendaciones["item_code"] = pd.to_numeric(df_recomendaciones["item_code"])
df_recomendaciones["score"] = pd.to_numeric(df_recomendaciones["score"])

In [ ]:
df_recomendaciones.dtypes

In [ ]:
df_recomendaciones

In [ ]:
tickets=pd.read_csv("Data/maestroestr.csv")

In [ ]:
tickets.dtypes

In [ ]:
tickets.rename(columns={'cod_est': 'item_code'}, inplace=True)
tickets

In [ ]:
merge2=pd.merge(tickets,df_recomendaciones, on="item_code", how="inner")
merge2

Medir el error (es ALS implicito)

In [ ]:
metricas=ranking_metrics_at_k(model, train_matrix, test_matrix, K=10) #k 10 para en geenral
print(metricas)